# Python 101 - Solutions
## Chapter XI

---

**For teaching assistants.** This notebook mirrors the exercises in
`../python101_11.ipynb` one for one. Most solutions end with an `assert`, so
running the whole notebook top to bottom is also a self-test: if it runs clean,
every solution still works.

There is usually more than one right answer - if a student's version passes the
same `assert`, it is correct.

In [ ]:
# Run from the chapter folder, so that `helpers`, `./data/...` and `./pics/...`
# resolve exactly the way they do in the lecture notebooks.
import os
import sys

if os.path.basename(os.getcwd()) == 'solutions':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

print('working directory:', os.getcwd())

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

BASE_URI = "./data/"
countries = ['United Kingdom', 'Hungary', 'France', 'Germany']

First rebuild the frames the notebook builds along the way, so the exercises have something to work on.

In [ ]:
# population
df = pd.read_csv(BASE_URI + 'population.csv')
columns = ['Country Name'] + [str(year) for year in range(1960, 2013)]
pop = (df[columns].dropna()
       .rename(columns={'Country Name': 'country'})
       .set_index('country'))
subpop = pop.loc[pop.index.isin(countries)].transpose()
subpop.index = subpop.index.astype('int')

# alcohol
df = pd.read_csv(BASE_URI + 'alcohol.csv')
columns = {'Country': 'country', 'Year': 'year',
           'Beverage Types': 'type', 'Display Value': 'alcohol'}
alc = df[list(columns.keys())].rename(columns=columns).dropna()

subalc = alc.loc[alc['country'].isin(countries)]
subalc = (subalc.loc[subalc['type'] == 'All']
          .drop(columns=['type'])
          .pivot(index='year', columns='country', values='alcohol'))

# trim both to the overlapping years and join
subpop = subpop.loc[(1989 < subpop.index) & (subpop.index < 2011)]
subalc = subalc.loc[(1989 < subalc.index) & (subalc.index < 2011)]
merged = subpop.join(subalc, rsuffix='_alc')

print('beverage types :', sorted(alc['type'].unique()))
print('years          :', alc.year.min(), '-', alc.year.max())
print('countries      :', alc.country.nunique())
merged.head()

### 1. The top 5 alcohol consuming countries in 1990

The `type` column has to be filtered to `'All'` first, otherwise beer, wine and spirits rows compete with the totals. Hungary comes first.

In [ ]:
alc_1990 = alc.loc[(alc['year'] == 1990) & (alc['type'] == 'All')]
top5_1990 = alc_1990.sort_values('alcohol', ascending=False).head(5)

print(top5_1990[['country', 'alcohol']].to_string(index=False))
top5_1990.set_index('country')['alcohol'].plot(
    kind='bar', title='Litres of pure alcohol per capita, 1990');

assert list(top5_1990['country'])[0] == 'Hungary'
assert len(top5_1990) == 5
assert top5_1990['alcohol'].is_monotonic_decreasing

### 2. Compare the average consumption of the four countries

`subalc` already has one column per country, so `.mean()` gives one average per country. These are the 1990-2010 averages, because Act III cut `subalc` to those years.

In [ ]:
averages = subalc.mean().sort_values(ascending=False)

print(averages.round(2).to_string())
averages.plot(kind='bar', title='Mean yearly consumption 1990-2010');

assert len(averages) == 4
assert averages.index[0] == 'France'
assert averages.index[-1] == 'United Kingdom'

# the same picture over time rather than as one number
subalc[countries].plot(figsize=(11, 5), title='...and how it changed');

### 3. The most spirit-consuming country each year

The sort-group-first steps from chapter X: after sorting by `alcohol`, `.first()` keeps the top country of every year. The Russian Federation came first most often, in 7 years.

In [ ]:
spirits = alc.loc[alc['type'] == 'Spirits']
winners = spirits.sort_values('alcohol', ascending=False).groupby('year').first()

print(winners[['country', 'alcohol']].to_string())
print()
print(winners['country'].value_counts().to_string())

assert len(winners) == spirits['year'].nunique()
assert winners.loc[1990, 'country'] == 'Kazakhstan'
assert winners['country'].value_counts().index[0] == 'Russian Federation'

### 4. How the different drinks changed in Hungary

A plain `pivot`, as in Act II: Hungary has exactly one row per year and type. Beer fell the most, from 6.59 to 3.91 litres between 1990 and 2010.

In [ ]:
hungary = (alc
           .loc[alc['country'] == 'Hungary']
           .pivot(index='year', columns='type', values='alcohol'))

print(hungary.tail().round(2).to_string())
hungary.plot(figsize=(11, 5), title='Hungary: alcohol consumption by type');

change = hungary.loc[2010] - hungary.loc[1990]
print('\nchange 1990 -> 2010:')
print(change.round(2).to_string())

assert 'Beer' in hungary.columns and 'Wine' in hungary.columns
# the total should be close to the sum of the parts
parts = [c for c in hungary.columns if c != 'All']
assert (hungary['All'] - hungary[parts].sum(axis=1)).abs().max() < 1.0
# beer fell the most
assert change[parts].sort_values().index[0] == 'Beer'

### 5. Whose consumption changed the most between 1990 and 2010?

`np.abs()` because we want the **biggest change in either direction** - and it matters: of the top five, three went down and two went up. Sorting on the raw difference would have given a completely different answer.

In [ ]:
totals = (alc
          .loc[alc['type'] == 'All']
          .pivot(index='country', columns='year', values='alcohol'))

totals['change'] = totals[2010] - totals[1990]
totals['abs_change'] = np.abs(totals['change'])
top5 = totals.sort_values('abs_change', ascending=False).head(5)

print('biggest change, either direction:')
print(top5['change'].round(2).to_string())
top5['change'].plot(kind='barh', title='Change in consumption, 1990 to 2010');

assert list(top5.index)[0] == 'Bosnia and Herzegovina'
assert round(top5.loc['Bosnia and Herzegovina', 'change'], 2) == -7.56
# three of the five fell, two rose - which is why we needed np.abs
assert len(top5.loc[top5['change'] > 0]) == 2
assert len(top5.loc[top5['change'] < 0]) == 3